# Kanji Detector — Treino YOLOv26n
> Dataset: `kanji-detector-dataset` | Modelo: `yolo26n.pt` | Epochs: 50

**Checklist antes de rodar:**
1. Adicionar o dataset `kanji-detector-dataset` como *Input* do notebook
2. Adicionar o dataset com o repositorio (ou clonar via Git abaixo)
3. Habilitar GPU: *Session options → Accelerator → GPU T4 x2 ou P100*

In [ ]:
# Verificar GPU disponivel
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout if result.returncode == 0 else 'GPU nao encontrada!')

import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA disponivel: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

## 1. Instalar dependencias

In [ ]:
!pip install -q ultralytics albumentations fonttools Pillow opencv-python-headless pyyaml requests
print('Dependencias instaladas.')

## 2. Configurar repositorio

> **Opcao A** (recomendado): clonar do GitHub  
> **Opcao B**: montar como dataset do Kaggle

In [ ]:
import os

# ── Opcao A: clonar do GitHub ─────────────────────────────────────────────
REPO_URL = 'https://github.com/MiguelMussalam/Detector-de-kanjis-n1.git'  # ajuste se necessario
WORK_DIR = '/kaggle/working'
REPO_DIR = os.path.join(WORK_DIR, 'Detector-de-kanjis-n1')

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    print(f'Repositorio ja existe em {REPO_DIR}')

os.chdir(REPO_DIR)
print(f'Diretorio atual: {os.getcwd()}')

# Adicionar ao sys.path para imports funcionarem
import sys
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
print('sys.path configurado.')

## 3. Verificar dataset de entrada

In [ ]:
import os

# Nome do dataset no Kaggle (deve coincidir com KAGGLE_DATASET em config.py)
KAGGLE_DATASET_NAME = 'kanji-detector-dataset'
DATASET_INPUT_DIR   = f'/kaggle/input/{KAGGLE_DATASET_NAME}'

if not os.path.exists(DATASET_INPUT_DIR):
    raise FileNotFoundError(
        f'Dataset nao encontrado em: {DATASET_INPUT_DIR}\n'
        'Adicione o dataset como Input do notebook no painel lateral.'
    )

# Contar imagens
train_imgs = os.listdir(os.path.join(DATASET_INPUT_DIR, 'images', 'train'))
val_imgs   = os.listdir(os.path.join(DATASET_INPUT_DIR, 'images', 'val'))
train_lbls = os.listdir(os.path.join(DATASET_INPUT_DIR, 'labels', 'train'))
val_lbls   = os.listdir(os.path.join(DATASET_INPUT_DIR, 'labels', 'val'))

print(f'Dataset encontrado: {DATASET_INPUT_DIR}')
print(f'  Train: {len(train_imgs)} imagens | {len(train_lbls)} labels')
print(f'  Val:   {len(val_imgs)} imagens  | {len(val_lbls)} labels')

## 4. Gerar dataset.yaml para o ambiente Kaggle

In [ ]:
import yaml, os

KAGGLE_DATASET_NAME = 'kanji-detector-dataset'
DATASET_INPUT_DIR   = f'/kaggle/input/{KAGGLE_DATASET_NAME}'
YAML_PATH           = '/kaggle/working/dataset.yaml'

config = {
    'path':  DATASET_INPUT_DIR,
    'train': 'images/train',
    'val':   'images/val',
    'nc':    1,
    'names': {0: 'kanji'},
}

with open(YAML_PATH, 'w') as f:
    yaml.dump(config, f, default_flow_style=False)

print(f'dataset.yaml gerado em: {YAML_PATH}')
print(open(YAML_PATH).read())

## 5. Carregar modelo yolo26n

O arquivo `yolo26n.pt` deve estar na raiz do repositorio.  
Se nao estiver, a ultralytics tentara baixar automaticamente.

In [ ]:
import os
from ultralytics import YOLO

MODEL_FILE = 'yolo26n.pt'
model_path = os.path.join(os.getcwd(), MODEL_FILE)

if os.path.exists(model_path):
    print(f'Modelo encontrado: {model_path}')
else:
    print(f'Modelo nao encontrado localmente. Usando: {MODEL_FILE}')
    model_path = MODEL_FILE

model = YOLO(model_path)
print(f'Modelo carregado: {model.info()}')

## 6. Treinar

In [ ]:
from ultralytics import YOLO
import os

# Hiperparametros
EPOCHS   = 50
IMGSZ    = 640
BATCH    = 16   # reduza para 8 se houver OOM no T4
WORKERS  = 2    # Kaggle: maximo 2 vCPUs
PROJECT  = 'kanji_detector'
RUN_NAME = 'run1'

model_path = os.path.join(os.getcwd(), 'yolo26n.pt')
if not os.path.exists(model_path):
    model_path = 'yolo26n.pt'

model = YOLO(model_path)

results = model.train(
    data      = '/kaggle/working/dataset.yaml',
    epochs    = EPOCHS,
    imgsz     = IMGSZ,
    batch     = BATCH,
    workers   = WORKERS,
    project   = PROJECT,
    name      = RUN_NAME,
    exist_ok  = True,
    save      = True,
    device    = 0,       # GPU 0
    patience  = 15,      # early stopping se mAP nao melhorar em 15 epochs
    cache     = True,    # cacheia imagens na RAM para acelerar treino
    amp       = True,    # mixed precision (mais rapido na GPU)
)

print(f'Treino concluido! Resultados em: {results.save_dir}')

## 7. Visualizar resultados

In [ ]:
from IPython.display import Image, display
import os

save_dir = results.save_dir
print(f'Resultados salvos em: {save_dir}')
print()

# Curvas de treino
results_plot = os.path.join(save_dir, 'results.png')
if os.path.exists(results_plot):
    print('=== Curvas de treino ===')
    display(Image(results_plot))

# Matriz de confusao
conf_matrix = os.path.join(save_dir, 'confusion_matrix.png')
if os.path.exists(conf_matrix):
    print('=== Matriz de confusao ===')
    display(Image(conf_matrix))

# Amostras de validacao
val_batch = os.path.join(save_dir, 'val_batch0_pred.jpg')
if os.path.exists(val_batch):
    print('=== Predicoes no conjunto de validacao ===')
    display(Image(val_batch))

## 8. Exportar modelo final

O `best.pt` e o checkpoint com melhor mAP. Faca download dele para usar localmente.

In [ ]:
import os, shutil

save_dir  = results.save_dir
best_pt   = os.path.join(save_dir, 'weights', 'best.pt')
last_pt   = os.path.join(save_dir, 'weights', 'last.pt')
output_dir = '/kaggle/working/output'
os.makedirs(output_dir, exist_ok=True)

for src in [best_pt, last_pt]:
    if os.path.exists(src):
        dst = os.path.join(output_dir, os.path.basename(src))
        shutil.copy2(src, dst)
        size_mb = os.path.getsize(dst) / 1e6
        print(f'Copiado: {dst}  ({size_mb:.1f} MB)')

# Tambem exportar para ONNX (opcional, descomente se quiser)
# from ultralytics import YOLO
# model_best = YOLO(best_pt)
# model_best.export(format='onnx', imgsz=640)
# print('Modelo exportado para ONNX.')

print(f'\nArquivos em /kaggle/working/output/:')
for f in os.listdir(output_dir):
    print(f'  {f}')